## **Imports**

In [12]:
import sys
import os

# Add the Marcelo directory to sys.path to allow merging of 'src' namespace packages.
# This ensures that Python sees both the root 'src' (contract) and 'Marcelo/src' (implementation).
marcelo_dir = os.path.abspath("Marcelo")
if marcelo_dir not in sys.path:
    sys.path.append(marcelo_dir)

from src.millionaire_client import AuthenticationError, MillionaireClient
from src.benchmark import Benchmark
from dotenv import load_dotenv
from src.models import ExperimentConfig, ApproachType
from src.guesser.marcelo_guesser import MarceloGuesser

## **Auth**

In [13]:
load_dotenv(dotenv_path=os.path.join("Marcelo", ".env"))


True

In [14]:
API_URL = "http://131.175.15.22:51111/"

USERNAME = os.getenv("MILLIONAIRE_USERNAME")
PASSWORD = os.getenv("MILLIONAIRE_PASSWORD")


In [15]:
client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


Welcome, MTKY! (Role: student)

=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)
  4: Philosophy and Psychology (15 questions)
  5: News (15 questions)


In [16]:
from src.guesser.engine.configs import INFERENCE_MODEL, EMBEDDING_MODEL

# "text" for normal mode, "speech" for audio transcription mode
MODE = "text"
TRANSCRIPTION_MODEL = "tiny"  # whisper model: tiny, base, small, medium, large

marcelo_experiment_config = ExperimentConfig(
    username="Marcelo",
    notes="Multi-theme dynamic approach test",
    approach=ApproachType.HYBRID,
    inference_model=INFERENCE_MODEL,
    inference_model_size="2b",
    embedding_model=EMBEDDING_MODEL,
    embedding_model_size="0.2b",
    is_rag=True,
    mode=MODE,
    transcription_model=TRANSCRIPTION_MODEL,
)

guesser = MarceloGuesser(
    marcelo_experiment_config,
    embedding_model_name=EMBEDDING_MODEL,
    inference_model_name=INFERENCE_MODEL,
)


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
 CRITICAL VRAM WARNING: The following models are running on CPU:
  - qwen2-math:1.5b
  - llama3.2:latest
  - qwen2.5:0.5b

 This will cause EXTREME slowness and likely TIMEOUTS.
 Your GPU VRAM is likely full. Consider using smaller models.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



In [17]:
benchmark = Benchmark(marcelo_experiment_config, guesser, client)
benchmark.run(1, filename="marcelo_benchmark_results.xlsx")

Preloading models...
Started game in competition: Entertainment (Mode: text)

--- Level 1 ---
Q: How does the film The Babadook relate to Jennifer Kent's previous work?
  [0] It is an extension of her short film Monster
  [1] It is a remake of her previous film
  [2] It is a short film she directed
  [3] It is unrelated to her previous works

Guesser is thinking...
 [Translator] Raw: "How does the film The Babadook relate to Jennifer ..." -> Keywords: "The Babadook"
 [Retriever] No chunks passed similarity threshold (>= 0.82). Using LLM's internal knowledge.
[RAG] Total: 11.94s | Search: 6.10s | Reasoning: 5.84s
Guesser chose option: 2 (Time: 13.59s, Transcription: 0.00s, Search: 6.10s, Reasoning: 5.84s)
GAME OVER! Result: INCORRECT
Preloading models...
Started game in competition: Ancient History and Politics (Mode: text)

--- Level 1 ---
Q: What term describes the practice of writing history that focuses on moral and ethical aspects, often to justify the actions of a particular polit